# Planck Spectroscopy Routine for JPA and LKIPA Noise Calibration
---

We ramp up the MXC Heater power until the temperature is stabilized, then keep the heater on at the same power and run the planck spec scripts of the JPA and the LKIPA

- $\sim 40$ data points from $10 mK - 200 mK$
- $\sim 20$ data points from $200 mK - 700 mK$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
import os
import h5py
import inspect
from tqdm import tqdm
import sys
import math
import presto
from presto import lockin, utils, hardware
from presto.hardware import AdcFSample, AdcMode, DacFSample, DacMode
from blueftc.BlueforsController import BlueFTController
# Reload credentials module to get latest changes
import importlib
import power_ramp as pr
importlib.reload(pr)

# Timeout for hardware communication
import requests

# BASE URL for interacting with BlueFTC
base_url = 'http://192.168.88.25:5001'

# Print JSON 
def print_JSON(r):
    r_dict = r.json()
    for key, value in r_dict.items():
        print(f"{key}: {value}")

## 1. MXC Heater Power List for Temperature Ramp Up

In [ ]:
MXC_Heater_Power_list_low_temp = 10 * np.arange(    # from 10mk - 100mk, in steps of 10 mu W
    start= 0,
    stop = 31,
    step = 1,
) 

MXC_Heater_Power_list_medium_temp = MXC_Heater_Power_list_low_temp[-1] + 50 * np.arange(    # from 100mk - 200mk, in steps of 50 mu W
    start= 1,
    stop = 15,
    step = 1,
) 

MXC_Heater_Power_list_high_temp = MXC_Heater_Power_list_medium_temp[-1] + 200 * np.arange(    # from 200mk - 700mk, in steps of 50 mu W
    start= 1,
    stop = 21,
    step = 1,
) 

MXC_Heater_Power_list = 1e-6 * np.concatenate(( # mu W
    MXC_Heater_Power_list_low_temp, 
    MXC_Heater_Power_list_medium_temp, 
    MXC_Heater_Power_list_high_temp
))

print((MXC_Heater_Power_list) * 1e6, 'muW') # print in mu W

# 1.1 Heater power for temperature cycle - only LKIPA planck

Temperature cycle: $10mK \to 50mK \to 200mK \to 50mK \to 10 mK$

In [ ]:
# MXC_Heater_cycle = 1e-6 * np.array([ # mu W
#     0,      # 10 mK 
#     70,     # 50 mK
#     1000,   # 200 mK
#     70      # 50 mK
# ])

# cycles = 4

# MXC_Heater_Power_list = np.tile(MXC_Heater_cycle, cycles)
# print((MXC_Heater_Power_list) * 1e6, 'muW') # print in mu W

## 2. Ramp up power and capture measurements for each temperature

In [ ]:
pr.heater_ramp_up(
    MXC_heater_power_list=MXC_Heater_Power_list,
    N_runs=100
)